# NTX 16 — Patched Native Executor and Certificate Security

CPU only. After Antigravity patches the repo, set NTX_RUN_NATIVE_TESTS=1 and NTX_REPO_REF to the patch branch. No reference harness passes substituted.

## Execution rules

This tests **source-native security**, not a self-contained reference broker. The known vulnerable commit is blocked. Antigravity must patch the repository and add real exploit-regression tests before a PASS claim is possible. Existing pytest success alone is insufficient.

In [ ]:
from pathlib import Path
import sys, os, csv, json, hashlib, zipfile, subprocess, time
import pandas as pd
from datetime import datetime, timezone
KIT = Path('/content/NiyamTraceX_Paper_Final_Validation_Kit') if Path('/content').exists() else Path.cwd().parent
if not (KIT/'evidence').is_dir():
    here = Path.cwd()
    choices = [here, here.parent, Path('/mnt/data/NiyamTraceX_Paper_Final_Validation_Kit')]
    for candidate in choices:
        if (candidate/'evidence').is_dir(): KIT = candidate; break
if not (KIT/'evidence').is_dir():
    package = None
    for p in [Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip',Path('/content/NiyamTraceX_Paper_Final_Validation_Kit.zip')]:
        if p.exists(): package=p;break
    if package is None and 'google.colab' in sys.modules:
        from google.colab import files
        print('Upload NiyamTraceX_Paper_Final_Validation_Kit.zip (the full kit).')
        upload = files.upload()
        if 'NiyamTraceX_Paper_Final_Validation_Kit.zip' in upload:
            package = Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip'
    if package:
        with zipfile.ZipFile(package) as z:
            for part in z.infolist():
                target=(Path('/content') if Path('/content').exists() else Path.cwd())/part.filename
                if not str(target.resolve()).startswith(str((Path('/content') if Path('/content').exists() else Path.cwd()).resolve())):
                    raise RuntimeError('Unsafe archive path')
            z.extractall('/content' if Path('/content').exists() else '.')
        KIT = (Path('/content') if Path('/content').exists() else Path.cwd())/'NiyamTraceX_Paper_Final_Validation_Kit'
OUT = Path(os.getenv('NTX_PAPER_OUT',str(KIT/'outputs')))
OUT.mkdir(parents=True,exist_ok=True)
print('Kit:',KIT,'\nOutput:',OUT)
assert (KIT/'evidence').is_dir(), 'Evidence not found. Upload the full kit ZIP.'
def dump(obj,p):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

In [ ]:
KNOWN='651e87ef452c80008d4f73c8f2de21321e21fa50'
REPO_URL='https://github.com/bnssaanirudh/NiyamTrace-X.git'
REPO_REF=os.environ.get('NTX_REPO_REF','main')
RUN_NATIVE_TESTS=os.environ.get('NTX_RUN_NATIVE_TESTS','0')=='1' # Set =1 after Antigravity produces a fixed commit
REPO=Path(os.getenv('NTX_REPO_DIR',str(KIT/'repo')))
ARCHIVED=json.loads((KIT/'evidence'/'oct10_native_archive.json').read_text())
old=[r for r in ARCHIVED.get('records',[]) if r.get('scenario') in ('direct_executor_bypass','untrusted_self_sealed_cert','missing_year_natural_pipeline')]
dump({'known_vulnerable_commit':KNOWN,'archive_records':old},OUT/'16_archived_native_counterexamples.json')
print('Historic native counterexamples:',[(r['scenario'],r['status'],r.get('mutated_rows')) for r in old])
print('RUN_NATIVE_TESTS:',RUN_NATIVE_TESTS,'ref:',REPO_REF)

In [ ]:
import shlex
record={'run_attempted':RUN_NATIVE_TESTS,'source_commit':None,'mandatory_native_probe_status':'NOT_RUN','tests':[],'status':'PENDING'}
if RUN_NATIVE_TESTS:
    if not (REPO/'.git').is_dir():
        subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO)],check=True,timeout=240)
    subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',REPO_REF],check=True,timeout=180)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True,timeout=90)
    commit=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    record['source_commit']=commit
    if commit==KNOWN: raise RuntimeError('Unpatched vulnerable commit; do not claim closure. Patch in Antigravity and push branch first.')
    package=REPO/'niyamtrace'
    assert (package/'packages/gate/executor.py').is_file()
    # Run test install in the current isolated Colab Python (no secret keys required).
    install=subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(package)+'[dev,postgres]','greenlet'],capture_output=True,text=True,timeout=600)
    (OUT/'16_install_stdout.log').write_text(install.stdout)
    (OUT/'16_install_stderr.log').write_text(install.stderr)
    if install.returncode!=0:raise RuntimeError('Dependency install failed; see logs')
    tests=['tests/security/test_native_mediation_regressions.py','tests/security/test_e2e_broker.py',
           'tests/integration/test_simulate_execute_atomicity.py']
    for test in tests:
        dest=package/test
        if not dest.is_file():
            record['tests'].append({'path':test,'status':'MISSING','returncode':None})
            continue
        env=os.environ.copy();env['PYTHONPATH']=str(package)+os.pathsep+env.get('PYTHONPATH','')
        p=subprocess.run([sys.executable,'-m','pytest','-q',test],cwd=package,env=env,capture_output=True,text=True,timeout=300)
        logname=test.replace('/','_')
        (OUT/f'16_{logname}.log').write_text(p.stdout+'\nSTDERR:\n'+p.stderr)
        record['tests'].append({'path':test,'status':'PASS' if p.returncode==0 else 'FAIL','returncode':p.returncode,'log_sha256':sha(OUT/f'16_{logname}.log')})
else:
    print('Patched commit test run disabled. Set NTX_RUN_NATIVE_TESTS=1 after source changes in GitHub.')

In [ ]:
# This cell verifies structured native evidence exported by the new repo-native regression runner.
# It does NOT trust arbitrary JSON flags as sufficient proof: logs and source must also be reviewed.
EVIDENCE_PATH=Path(os.getenv('NTX_NATIVE_EVIDENCE_JSON',str(REPO/'niyamtrace'/'artifacts'/'PRODUCTION_FIX_EVIDENCE.json')))
required={'SEC-01_DIRECT_EXECUTOR_MEDIATION','SEC-02_CERTIFICATE_ORIGIN_BINDING','SEC-03_MISSING_YEAR',
          'SEC-04_PAIRED_BROKER_POSITIVE','SEC-REPLAY','SEC-STALE_STATE','SEC-WRONG_TENANT','SEC-POLICY_REVOKE'}
issues=[]
if not RUN_NATIVE_TESTS:issues.append('Patched native tests not executed')
if record['source_commit'] is None:issues.append('No checked-out commit')
if not EVIDENCE_PATH.is_file(): issues.append('No PRODUCTION_FIX_EVIDENCE.json from source-native test runner')
else:
    j=json.loads(EVIDENCE_PATH.read_text())
    if j.get('repo_commit')!=record['source_commit']:issues.append('Repo HEAD / evidence revision mismatch')
    got={x.get('id'):x for x in j.get('probes',[]) if isinstance(x,dict)}
    for key in sorted(required):
        v=got.get(key)
        if v is None:issues.append('Missing '+key);continue
        if v.get('status')!='PASS' or v.get('actual_native_executor') is not True:issues.append('Not verified '+key)
        for d in ('trace_sha256','before_db_sha256','after_db_sha256'):
            if not isinstance(v.get(d),str) or len(v[d])!=64:issues.append(key+' missing '+d)
        if key in {'SEC-01_DIRECT_EXECUTOR_MEDIATION','SEC-02_CERTIFICATE_ORIGIN_BINDING','SEC-03_MISSING_YEAR'} and v.get('rows_mutated')!=0:
            issues.append(key+' still mutated rows')
        if key=='SEC-04_PAIRED_BROKER_POSITIVE' and not (isinstance(v.get('rows_mutated'),int) and v['rows_mutated']>0):
            issues.append('Positive authorized commit did not succeed')
if record['tests'] and any(x['status']!='PASS' for x in record['tests']):issues.append('Regression tests missing/failed')
record['mandatory_native_probe_status']='CANDIDATE_PASS_REQUIRES_INDEPENDENT_REVIEW' if not issues else 'BLOCKED'
record['issues']=issues
record['status']='CANDIDATE_REVIEW' if not issues else 'PENDING_OR_FAIL'
dump(record,OUT/'16_PRODUCTION_SECURITY_GATE.json')
print(json.dumps(record,indent=2)[:5000])